<a href="https://colab.research.google.com/github/rushikesh-D69/water/blob/main/habitability_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Adaptive AI-Driven Telescope Target Prioritization Framework
## Stage 1: Probabilistic Exoplanet Prioritization

**Paper Title:** Adaptive Reinforcement Learning-Based Telescope Target Prioritization for Probabilistic Liquid Water Detection on Exoplanets

---

### What This Notebook Does

This is NOT a habitability classifier. It is a scientific prioritization system that answers:

> Which exoplanets should a telescope observe NEXT to maximize scientific return per observation hour?

---

### Key Design Decisions (v2.0)

| Decision | Old (v1) | New (v2) | Reason |
|----------|----------|----------|--------|
| Task | Binary classification | Continuous regression / ranking | Prioritization is not binary |
| Data leakage | HZ/ESI used as features | HZ/ESI used only for label | Prevents circular learning |
| Target | habitable = 0/1 | priority_score in [0,1] | Ranking requires ordinal target |
| Observation | Not modelled | SNR proxy, detectability, distance | Real telescope scheduling |
| Uncertainty | None | Ensemble variance per planet | Research-grade uncertainty quantification |
| Scientific gain | None | uncertainty * detectability | Information-theoretic scheduling |
| Metrics | ROC-AUC, F1 | NDCG, MAP, Spearman rho, Kendall tau | Ranking-appropriate evaluation |
| Simulation | None | Temporal observation rounds | Motivates Stage 2 RL scheduler |

---

### Scientific Formulations

**Habitable Zone (Kopparapu et al. 2014) -- used for label only:**

$$S_{eff} = S_{eff\odot} + aT_* + bT_*^2 + cT_*^3 + dT_*^4, \quad T_* = T_{eff} - 5780 \text{ K}$$

**Priority Score (weak supervision target):**

$$\text{priority\_score} = f_{\text{thermal}}^{0.5} \cdot f_{\text{rocky}}^{0.3} \cdot f_{\text{ESI}}^{0.2}$$

**Scientific Gain (observation value metric):**

$$\text{ScientificGain}_i = \sigma_i \times \text{Detectability}_i$$

**Ranking Metrics:**

$$\text{NDCG@K} = \frac{DCG@K}{IDCG@K} = \frac{\sum_{i=1}^{K} \text{score}_i / \log_2(i+1)}{\text{ideal}}$$

---
## 0. Environment Setup

This cell handles both **Google Colab** (clones from GitHub) and **local** environments automatically.

In [ ]:
import sys, os

# ── Detect environment ─────────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # ── GitHub config ─────────────────────────────────────────────────────
    GITHUB_USER = 'rushikesh-D69'   # already set
    REPO_NAME   = 'water'           # already set
                  # <-- change this if different
    BRANCH      = 'main'

    REPO_URL = f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git'

    print(f'[Colab] Cloning {REPO_URL} ...')
    os.system(f'git clone {REPO_URL} /content/{REPO_NAME}')
    os.chdir(f'/content/{REPO_NAME}')

    print('[Colab] Installing dependencies ...')
    os.system('pip install -q xgboost lightgbm shap scipy scikit-learn '
              'matplotlib seaborn requests joblib')
    print('[Colab] Setup complete.')
else:
    # ── Local: just ensure project root is in path ─────────────────────────
    ROOT = os.path.abspath('.')
    if ROOT not in sys.path:
        sys.path.insert(0, ROOT)
    print(f'[Local] Project root: {ROOT}')

---
## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import xgboost as xgb
import joblib
from scipy.stats import spearmanr, kendalltau
from IPython.display import Image, display

from src.data_acquisition import (
    fetch_nasa_exoplanets, run_pipeline,
    ML_FEATURES, TARGET, DATA_DIR, PLOTS_DIR
)
from src.ml_pipeline import (
    load_data, run_ml_pipeline,
    compute_ranking_metrics, ndcg_at_k, map_at_k, regret_at_k
)

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
plt.rcParams['figure.dpi'] = 100
print('All imports OK')

---
## 2. Data Acquisition -- NASA Exoplanet Archive

Fetches from `pscomppars` (Planetary Systems Composite Parameters) via TAP API.
Cached locally after first download.

In [ ]:
# Run the full data pipeline
# force_refresh=True to re-download from NASA
df_ml, feature_names = run_pipeline(force_refresh=False)

print(f'Dataset: {df_ml.shape}')
print(f'Features: {len(feature_names)}')
print(f'Target: {TARGET} (continuous [0,1])')

In [ ]:
# Preview raw data
preview_cols = ['pl_name', 'hostname', 'pl_rade', 'pl_bmasse', 'pl_eqt',
                'pl_orbsmax', 'st_teff', 'detectability', 'priority_score']
df_ml[preview_cols].head(10)

In [ ]:
# Verify NO data leakage: HZ/ESI must NOT appear in ML features
leakage_cols = ['hz_factor', 'esi', 'hz_rv', 'hz_rg', 'hz_mg', 'hz_em',
                '_hz_factor_diag', '_esi_diag', '_rocky_diag']
leak_found = [c for c in leakage_cols if c in feature_names]

if leak_found:
    print(f'WARNING - Leakage detected: {leak_found}')
else:
    print('No leakage. HZ/ESI correctly excluded from ML features.')
    print(f'\nML feature set ({len(feature_names)} features):')
    for i, f in enumerate(feature_names, 1):
        print(f'  {i:2d}. {f}')

---
## 3. Priority Score -- Target Label Analysis

The target is built via **weak supervision** using astrophysical theory.
Multiplicative form ensures all three conditions must hold simultaneously:

$$\text{priority\_score} = f_{\text{thermal}}^{0.5} \cdot f_{\text{rocky}}^{0.3} \cdot f_{\text{ESI}}^{0.2}$$

The label components ($f_{\text{thermal}}$, $f_{\text{rocky}}$, $f_{\text{ESI}}$) are excluded from ML features to prevent data leakage.

In [ ]:
Image(str(PLOTS_DIR / 'priority_score_distribution.png'), width=1000)

In [ ]:
ps = df_ml['priority_score']
print('Priority Score Statistics:')
print(f'  Mean:    {ps.mean():.4f}')
print(f'  Std:     {ps.std():.4f}')
print(f'  Max:     {ps.max():.4f}')
print(f'  Median:  {ps.median():.4f}')
print(f'  >= 0.30: {(ps >= 0.30).sum():,} planets ({100*(ps>=0.30).mean():.1f}%)')
print(f'  >= 0.20: {(ps >= 0.20).sum():,} planets ({100*(ps>=0.20).mean():.1f}%)')

In [ ]:
# Top 20 planets by raw priority score (ground truth from weak supervision)
top_raw = df_ml.nlargest(20, 'priority_score')[
    ['pl_name', 'hostname', 'priority_score', 'detectability',
     'pl_rade', 'pl_eqt', 'pl_orbsmax', 'st_teff', 'spectral_class']
].reset_index(drop=True)
top_raw.index += 1

print('Top 20 Planets by Priority Score (weak supervision label):')
display(top_raw.style
    .background_gradient(subset=['priority_score', 'detectability'], cmap='YlGn')
    .format({'priority_score': '{:.4f}', 'detectability': '{:.4f}',
             'pl_rade': '{:.2f}', 'pl_eqt': '{:.0f}', 'pl_orbsmax': '{:.3f}'}))

---
## 4. Observation Feasibility Analysis

Observation features are **real ML inputs** (not label components).
They model telescope scheduling cost/benefit:

| Feature | Physical Meaning |
|---------|------------------|
| `snr_proxy` | Signal-to-noise proxy for atmospheric spectroscopy |
| `distance_penalty` | Accessibility penalty: exp(-d / 100 pc) |
| `transit_depth_norm` | Normalised transit depth (fraction of starlight blocked) |
| `obs_duration_norm` | Normalised transit duration |
| `detectability` | Composite observability = 0.5*SNR + 0.3*dist_pen + 0.2*depth |

In [ ]:
Image(str(PLOTS_DIR / 'observability_analysis.png'), width=1100)

In [ ]:
# Planets with best combined priority + detectability
df_ml['obs_utility'] = df_ml['priority_score'] * df_ml['detectability']
top_obs = df_ml.nlargest(15, 'obs_utility')[
    ['pl_name', 'hostname', 'priority_score', 'detectability', 'obs_utility',
     'snr_proxy', 'sy_dist', 'pl_rade', 'st_teff']
].reset_index(drop=True)
top_obs.index += 1

print('Top 15 Planets by Observation Utility (priority * detectability):')
display(top_obs.style
    .background_gradient(subset=['obs_utility', 'priority_score', 'detectability'], cmap='plasma')
    .format({'priority_score': '{:.4f}', 'detectability': '{:.4f}',
             'obs_utility': '{:.4f}', 'snr_proxy': '{:.2f}',
             'sy_dist': '{:.1f}', 'pl_rade': '{:.2f}'}))

---
## 5. Feature Correlation Analysis

In [ ]:
Image(str(PLOTS_DIR / 'feature_correlations.png'), width=900)

In [ ]:
key_feats = feature_names + ['priority_score']
key_feats = [f for f in key_feats if f in df_ml.columns]
corr_with_target = df_ml[key_feats].corr()['priority_score'].drop('priority_score')
print('Feature Correlations with Priority Score (sorted by |correlation|):')
print(corr_with_target.abs().sort_values(ascending=False).head(20).round(4).to_string())

---
## 6. Machine Learning -- Probabilistic Prioritization Models

We train **regression** models to predict `priority_score` from raw astrophysical parameters.

Evaluation uses **ranking metrics** because the order matters, not the absolute score:
- Spearman rho and Kendall tau: rank correlation
- NDCG@K: quality of top-K recommendations
- MAP@K: precision of top-K selection
- Regret@K: scientific value missed by not selecting the ideal top-K

5-Fold CV is run on Spearman correlation (not R2) to measure ranking generalization.

In [ ]:
results, ranking, sim_log = run_ml_pipeline()

---
## 7. Ranking Performance Analysis

In [ ]:
Image(str(PLOTS_DIR / 'ranking_metrics.png'), width=900)

In [ ]:
Image(str(PLOTS_DIR / 'cv_spearman.png'), width=750)

In [ ]:
Image(str(PLOTS_DIR / 'predicted_vs_actual.png'), width=1100)

In [ ]:
metrics_rows = []
for name, res in results.items():
    m  = res['metrics']
    cv = res['cv_spearman']
    metrics_rows.append({
        'Model':              name,
        'NDCG@50':            m['NDCG@50'],
        'MAP@50':             m['MAP@50'],
        'Spearman rho':       m['Spearman'],
        'Kendall tau':        m['Kendall_Tau'],
        'Regret@50':          m['Regret@50'],
        'R2':                 m['R2'],
        'RMSE':               m['RMSE'],
        'CV Spearman (mean)': cv.mean(),
        'CV Spearman (std)':  cv.std()
    })

metrics_df = pd.DataFrame(metrics_rows).set_index('Model')
print('Full Ranking Metrics:')
display(metrics_df.style
    .background_gradient(subset=['NDCG@50', 'MAP@50', 'Spearman rho', 'Kendall tau'], cmap='YlGn')
    .background_gradient(subset=['Regret@50', 'RMSE'], cmap='YlOrRd_r')
    .format('{:.4f}'))

---
## 8. Uncertainty Estimation

Prediction uncertainty is the standard deviation across individual decision tree predictions:

$$\sigma_i = \text{std}\left(\{T_1(x_i),\, T_2(x_i),\, \ldots,\, T_N(x_i)\}\right)$$

Example output:
- Planet A: 0.82 +/- 0.05  (high confidence)
- Planet B: 0.81 +/- 0.24  (low confidence -- needs observation)

In [ ]:
Image(str(PLOTS_DIR / 'uncertainty_analysis.png'), width=1000)

In [ ]:
if 'Random Forest' in results:
    res = results['Random Forest']
    unc_df = res['meta_test'].copy()
    unc_df['pred_priority']   = res['uncertainty_mean']
    unc_df['uncertainty']     = res['uncertainty_std']
    unc_df['scientific_gain'] = res['scientific_gain']
    unc_df['pred_str'] = [
        f"{m:.3f} +/- {s:.3f}"
        for m, s in zip(res['uncertainty_mean'], res['uncertainty_std'])
    ]

    top_gain = unc_df.nlargest(10, 'scientific_gain')
    print('Top 10 by Scientific Gain (high uncertainty + high detectability):')
    print('These are the most INFORMATIVE targets to observe next.')
    display(top_gain[['pl_name', 'pred_str', 'detectability', 'scientific_gain']]
        .rename(columns={'pl_name': 'Planet', 'pred_str': 'Priority Score',
                         'detectability': 'Detectability', 'scientific_gain': 'Scientific Gain'})
        .reset_index(drop=True)
        .style.background_gradient(subset=['Scientific Gain'], cmap='plasma')
        .format({'Detectability': '{:.4f}', 'Scientific Gain': '{:.4f}'}))

---
## 9. Explainable AI -- Astrophysical Drivers of Prioritization

SHAP quantifies which measurable planetary/stellar properties most influence whether
a planet should be observed next.

Color coding:
- Green bars: Planetary parameters
- Gold bars: Stellar parameters
- Pink bars: Observational parameters

In [ ]:
rf_shap = PLOTS_DIR / 'shap_random_forest.png'
if rf_shap.exists():
    Image(str(rf_shap), width=900)
else:
    print('SHAP plot not generated yet.')

In [ ]:
xgb_shap = PLOTS_DIR / 'shap_xgboost.png'
if xgb_shap.exists():
    Image(str(xgb_shap), width=900)

In [ ]:
# Waterfall plot: explain a single planet's prioritization score
if 'XGBoost' in results:
    res = results['XGBoost']
    X_test   = res['X_test']
    y_pred   = res['y_pred']
    meta_t   = res['meta_test']

    best_idx    = np.argmax(y_pred)
    planet_name = meta_t['pl_name'].iloc[best_idx]
    print(f'SHAP Waterfall -- highest priority planet: {planet_name}')
    print(f'Predicted priority: {y_pred[best_idx]:.4f}')

    explainer = shap.TreeExplainer(res['model'])
    sv = explainer(X_test)
    shap.plots.waterfall(sv[best_idx], max_display=15)

In [ ]:
Image(str(PLOTS_DIR / 'feature_importance.png'), width=1100)

---
## 10. Temporal Observation Simulation

Simulates 3 rounds of telescope observations.
Each round selects top-10 targets ranked by (priority + 0.5 * scientific_gain).
After observation, uncertainty halves (information gained).

Compared against:
- Random selection baseline
- Static ML ranking (no uncertainty awareness)
- Our system (uncertainty-aware)

In [ ]:
Image(str(PLOTS_DIR / 'temporal_simulation.png'), width=1000)

In [ ]:
if sim_log:
    for row in sim_log:
        print(f"Round {row['round']}: gain={row['cum_sci_gain']:.4f} | "
              f"mean_priority={row['mean_priority']:.4f} | "
              f"targets: {', '.join(row['top_k_planets'][:5])} ...")

---
## 11. Final Priority Ranking

In [ ]:
ranking_path = DATA_DIR / 'final_priority_ranking.csv'
if ranking_path.exists():
    rank_df = pd.read_csv(ranking_path, index_col=0)
    print(f'Final ranking loaded: {len(rank_df):,} planets')

    top25 = rank_df.head(25)[[
        'pl_name', 'hostname', 'pred_str', 'scientific_gain',
        'detectability', 'pl_rade', 'pl_eqt', 'spectral_class'
    ]].copy()
    top25.columns = ['Planet', 'Star', 'Priority (mean +/- std)', 'Sci. Gain',
                     'Detectability', 'Radius (Re)', 'T_eq (K)', 'Spectral']

    print('\nTop 25 Telescope Observation Targets:')
    display(top25.style
        .background_gradient(subset=['Sci. Gain', 'Detectability'], cmap='YlGn')
        .format({'Sci. Gain': '{:.4f}', 'Detectability': '{:.4f}',
                 'Radius (Re)': '{:.2f}', 'T_eq (K)': '{:.0f}'}))

---
## 12. Push Results to GitHub (Colab only)

In [ ]:
if IN_COLAB:
    # Configure git (run once per session)
    GIT_EMAIL = 'rikki0501hanuman@gmail.com'           # <-- change this
    GIT_NAME  = 'rushikesh-D69'                # <-- change this

    os.system(f'git config user.email "{GIT_EMAIL}"')
    os.system(f'git config user.name "{GIT_NAME}"')

    # Add generated outputs and push
    os.system('git add data/ models/ plots/')
    os.system('git commit -m "Stage 1: Add processed data, models, and plots from Colab run"')

    # For private repos, use a Personal Access Token (PAT)
    # Set GITHUB_TOKEN as a Colab secret (Secrets panel on the left)
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    remote = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    os.system(f'git remote set-url origin {remote}')
    os.system(f'git push origin {BRANCH}')
    print('Pushed to GitHub.')
else:
    print('Local environment -- use git push manually.')

---
## 13. Stage 1 Summary

### Contributions

| Component | Description |
|-----------|-------------|
| Leakage-free pipeline | HZ/ESI used only for weak supervision labels, not ML inputs |
| Ranking formulation | Priority score regression, not binary classification |
| Observation modelling | SNR proxy, distance penalty, detectability as first-class features |
| Uncertainty quantification | Tree ensemble variance: priority = 0.82 +/- 0.11 |
| Scientific gain | ScientificGain = uncertainty * detectability |
| Ranking-aware metrics | NDCG@50, MAP@50, Spearman rho, Kendall tau, Regret@50 |
| Temporal simulation | 3-round simulation showing gain over random/static baselines |

### Research Questions Addressed
- **RQ1:** Can ML rank exoplanets better than random/static baselines? -- Measured via NDCG and Regret@K

### Next Stages
- **Stage 2:** Dynamic Prioritization Engine with adaptive weight tuning
- **Stage 3:** RL Autonomous Scheduler

In [ ]:
# Output file summary
from pathlib import Path
ROOT_PATH = Path('.')

print('Generated Output Files:')
print('\n  Data:')
for f in sorted(DATA_DIR.glob('*.csv')):
    print(f'    {f.name}  ({f.stat().st_size/1024:.1f} KB)')

print('\n  Models:')
for f in sorted((ROOT_PATH / 'models').glob('*')):
    print(f'    {f.name}  ({f.stat().st_size/1024:.1f} KB)')

print('\n  Plots:')
for f in sorted(PLOTS_DIR.glob('*.png')):
    print(f'    {f.name}')